# TFL Generation Agent — Official 6-Study Library (Real SAP / DeepSeek)

**Uses the real consolidated 431-template Library — not demo templates, not fixture LLM responses.**

**Target PDF verified against its TFL Content List:** `SAP_001_00.pdf` is the 106-page final SAP for study `RP6530-2101` (Tenalisib, Phase II breast cancer). Its TFL Content List is on **physical PDF pages 23–26**: 47 Tables, 44 Listings, and Figure entries. We select **8 verified Table targets** from that list (6 run by default) and **2 Listing targets** for *retrieval-only* experiments; Figures are excluded.

- **SAP:** read your existing PDF from private Google Drive; no source PDF is committed to GitHub.
- **Library:** official `library/` root with 431 registered source DOCX templates from six studies (208 Tables, 203 Listings, 20 Figures at the time of writing).
- **Table workflow:** Step 1 SAP parsing → Step 2 **live DeepSeek** evidence-based specification → Step 3 cross-study Library retrieval → Step 4 safe adaptation attempt → Step 5 validation/export.
- **Listing experiment:** metadata/title-based retrieval against all approved Listing source templates using the existing Step 3 ranking function. **No Listing specification extraction or DOCX generation:** the Agent's Step 2 currently supports Table only.
- **Known limitation:** the Agent on `main` cannot yet adapt authentic bookmark-based DOCX source templates without review. A real-source run may return `needs_review` with no output DOCX; this is not a completed shell.

**Privacy:** Transmit SAP excerpts to DeepSeek only if authorized. Do not commit private SAPs, keys, or output reports to this public GitHub repository. Original source Library is read-only.

## 0 · Install the current Agent from GitHub `main`

A fresh Colab runtime is recommended. This notebook itself can live on a feature branch, while the Agent and the **latest official Library** are always cloned from `main`. The clone is isolated from previous demo folders.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
PROJECT_ROOT = Path("/content/tfl-generation-agent-official")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.tools.llm_client import DeepSeekJSONClient
from tfl_agent.tools.io_utils import write_json
import tfl_agent
import pandas as pd
from IPython.display import display, FileLink

assert Path(tfl_agent.__file__).resolve().is_relative_to(PROJECT_ROOT), (
    "An older tfl_agent module is cached. Restart the runtime and rerun from the first cell."
)
commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Agent loaded from:", tfl_agent.__file__)
print("GitHub main commit:", commit)
print("Project path:", PROJECT_ROOT)

## 1 · Audit the **official unified Library** (no API call)

This reads the consolidated root `library/`, **not** `library/studies/P-VCNA-003`, `examples/`, or `input/synthetic_sap.pdf`.

The existing Agent loader exposes the root through compatibility aliases (`metadata/Template_Metadata.json` and `templates/`). Those aliases point to the **official** unified Metadata and canonical DOCX collection. We also use `Template_Index.json` to attach the authoritative **per-study bookmark locator paths** in memory. Nothing is written into `library/`.

Only `review_status=approved` and `formal_use_eligible=true` records are passed to retrieval. Source approval does not grant automatic cross-study adaptation.

In [ ]:
import json
from collections import Counter

LIBRARY_PATH = PROJECT_ROOT / "library"  # OFFICIAL unified six-study Library
REGISTRY_PATH = LIBRARY_PATH / "Library_Registry.json"
TEMPLATE_INDEX_PATH = LIBRARY_PATH / "Template_Index.json"
CANONICAL_DOCX_ROOT = (LIBRARY_PATH / "TFL_Template_Library").resolve()

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
template_index = json.loads(TEMPLATE_INDEX_PATH.read_text(encoding="utf-8"))
index_records = {r["template_id"]: r for r in template_index["entries"]}
approved_ids = {
    r["template_id"] for r in template_index["entries"]
    if r.get("review_status") == "approved" and r.get("formal_use_eligible") is True
}

official_library = load_library(LIBRARY_PATH)
loader_format = official_library["format"]  # May show 'mvp' due to the compatibility alias.
assert registry["counts"]["registered_templates"] == len(official_library["templates"])
assert registry["counts"]["approved"] == len(approved_ids)
assert len(official_library["templates"]) >= 400, (
    "The official consolidated Library did not load: check GitHub main and symbolic links."
)

# Resolve canonical DOCX and per-study locators from the official Template_Index,
# without modifying Metadata, source files, or Agent code.
missing_locators = []
for template in official_library["templates"]:
    tid = template["identity"]["template_id"]
    entry = index_records[tid]
    expected_docx = (PROJECT_ROOT / entry["docx_path"]).resolve()
    actual_docx = Path(template["resolved_path"]).resolve()
    assert expected_docx == actual_docx, f"DOCX path mismatch: {tid}"
    assert actual_docx.is_file() and actual_docx.is_relative_to(CANONICAL_DOCX_ROOT), tid
    locator = (PROJECT_ROOT / entry["locator_map_path"]).resolve()
    if locator.is_file():
        template["locator_map_path"] = str(locator)  # In-memory path only.
    else:
        template["locator_map_path"] = None
        missing_locators.append(tid)

official_library["templates"] = [
    template for template in official_library["templates"]
    if template["identity"]["template_id"] in approved_ids
]
official_library["library_id"] = registry["library_id"]
official_library["format"] = "official_unified_v2_4_approved"
assert len(official_library["templates"]) == len(approved_ids)
assert len(official_library["templates"]) > 1

counts = Counter(t["identity"]["output_type"] for t in official_library["templates"])
study_counts = Counter(t["source"]["study_id"] for t in official_library["templates"])
keyword_mapping = official_library["index"].get("keyword_to_template_ids", {})
assert keyword_mapping, "Expected official global Keyword Index, not an empty demo index."
assert set(study_counts) == {s["study_id"] for s in registry["studies"]}

print("OFFICIAL LIBRARY ID:", official_library["library_id"])
print("Approved / registered:", len(approved_ids), "/", registry["counts"]["registered_templates"])
print("Original compatibility loader label:", loader_format, "(not the source data type)")
print("Global indexed keywords:", len(keyword_mapping))
print("Per-study locator maps missing:", len(missing_locators))
display(pd.DataFrame([{"Type": k, "Approved templates": v} for k, v in sorted(counts.items())]))
display(pd.DataFrame([{"Study": k, "Approved templates": v} for k, v in sorted(study_counts.items())]))
if missing_locators:
    print("First missing locator IDs:", missing_locators[:10])

## 2 · Load your real target SAP from Google Drive

The default path below matches the previous real-study experiment. You may change the path and physical PDF page range for a different study.

For `SAP_001_00.pdf`, the default `(4, 21)` selects **SAP body pages only**, excluding later Content List / source shell pages. Do **not** reuse this page range for a different PDF without checking it. Upload your PDF to Google Drive once; it is not sent to GitHub.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_SAP_RELATIVE_PATH = "Colab Notebooks/agent/midterm demo/SAP_001_00.pdf"
SAP_PATH = Path("/content/drive/MyDrive") / DRIVE_SAP_RELATIVE_PATH
SAP_BODY_PAGE_RANGE = (4, 21)  # Physical PDF pages, 1-based; change for a different PDF.

if not SAP_PATH.is_file():
    folder = SAP_PATH.parent
    nearby = sorted(p.name for p in folder.glob("*.pdf")) if folder.exists() else []
    raise FileNotFoundError(
        f"Missing SAP at {SAP_PATH}. Folder exists: {folder.exists()}. "
        f"PDF names found: {nearby}. Change DRIVE_SAP_RELATIVE_PATH above."
    )
print("Real SAP:", SAP_PATH)
print("File size (MB):", round(SAP_PATH.stat().st_size / (1024 * 1024), 2))
print("Evidence page range:", SAP_BODY_PAGE_RANGE)

## 3 · Select real targets from the PDF's Content List

The eight **Table IDs and titles below are copied from the actual RP6530-2101 SAP** (PDF pages 23–24), not invented examples. They cover patient disposition, demographics, medical history, objective response, PFS, AEs, hematology, and vital signs.

**Six Tables run by default** for diverse evaluation; edit `RUN_TABLE_IDS` to use a smaller subset to control DeepSeek cost or to include all eight. Each Table can make two or three live model calls. In particular, PFS and ORR/CBR are more statistically complex than demographic summaries.

Two verified Listings from PDF page 25 will be used separately in a **retrieval-only** experiment: `16.2.4.1` (Demography Data) and `16.2.7.1` (Adverse Events). They are intentionally **not** passed to `TFLSession.step2_build_specification()`, which currently rejects Listing targets.

All TFL IDs are **target-SAP** IDs, not Library source-template IDs. The existing workflow handles *one Table per session* and does not assert complete-study coverage.

In [ ]:
# Verified against SAP_001_00.pdf, physical pages 23–24.
TABLE_TARGET_CATALOG = [
    {"tfl_id": "14.1.1.1", "title": "Patients Disposition - All Patients",
     "topic": "Disposition", "complexity": "basic"},
    {"tfl_id": "14.1.3.1", "title": "Demographic and Baseline Characteristics – Safety Analysis Set",
     "topic": "Demographics", "complexity": "basic"},
    {"tfl_id": "14.1.4.1", "title": "Summary of Medical and Surgical History – Safety Analysis Set",
     "topic": "Medical history / MedDRA", "complexity": "intermediate"},
    {"tfl_id": "14.2.2.1", "title": "Summary of Objective Response Rate and Clinical Benefit Rate - ITT Analysis Set",
     "topic": "ORR / CBR; binomial CI", "complexity": "advanced"},
    {"tfl_id": "14.2.3.1", "title": "Summary of Progression Free Survival - ITT Analysis Set",
     "topic": "PFS / Kaplan–Meier", "complexity": "advanced"},
    {"tfl_id": "14.3.1.1", "title": "Overall Summary of Adverse Events - Safety Analysis Set",
     "topic": "Adverse events", "complexity": "intermediate"},
    {"tfl_id": "14.3.2.1", "title": "Descriptive Summary of Hematology parameters - Safety Analysis Set",
     "topic": "Hematology", "complexity": "intermediate"},
    {"tfl_id": "14.3.3.1", "title": "Descriptive Summary of Vital Signs - Safety Analysis Set",
     "topic": "Vital signs", "complexity": "intermediate"},
]

# Default: six varied Tables. Change this to any subset of the catalog (1–8 IDs).
# To run all eight: RUN_TABLE_IDS = [t["tfl_id"] for t in TABLE_TARGET_CATALOG]
RUN_TABLE_IDS = [
    "14.1.1.1",  # Patient disposition
    "14.1.3.1",  # Demographics (previous real-SAP pilot)
    "14.2.2.1",  # ORR / CBR
    "14.2.3.1",  # PFS
    "14.3.1.1",  # Adverse events
    "14.3.2.1",  # Hematology
]

# Verified against SAP_001_00.pdf, physical page 25.
LISTING_TARGETS = [
    {"tfl_id": "16.2.4.1", "title": "Demography Data - Safety Analysis Set"},
    {"tfl_id": "16.2.7.1", "title": "Adverse Events - Safety Analysis Set"},
]
RUN_LISTING_RETRIEVAL_ONLY = True  # Metadata/title-only; no DeepSeek or DOCX.
LISTING_TOP_K = 5

catalog_by_id = {t["tfl_id"]: t for t in TABLE_TARGET_CATALOG}
assert len(catalog_by_id) == len(TABLE_TARGET_CATALOG)
assert 1 <= len(RUN_TABLE_IDS) <= len(TABLE_TARGET_CATALOG)
assert len(set(RUN_TABLE_IDS)) == len(RUN_TABLE_IDS)
assert all(tid in catalog_by_id for tid in RUN_TABLE_IDS), "Unknown selected Table ID"
TARGET_TABLES = [catalog_by_id[tid] for tid in RUN_TABLE_IDS]

TOP_K = 10
MIN_SCORE = 0.55
OUTPUT_DIR = PROJECT_ROOT / "output" / "official_library_deepseek"
SCHEMA_PATH = PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"

display(pd.DataFrame([
    {**t, "run": "yes" if t["tfl_id"] in RUN_TABLE_IDS else "optional"}
    for t in TABLE_TARGET_CATALOG
]))
print("Selected Table count:", len(TARGET_TABLES))
print("Selected Table IDs:", RUN_TABLE_IDS)
print("Optional Listings (retrieval-only):", [t["tfl_id"] for t in LISTING_TARGETS])
print("Official Library scope:", len(official_library["templates"]), "approved templates")
print("Output directory:", OUTPUT_DIR)

## 4 · Connect DeepSeek (real API — no fixture)

In Colab **Secrets** (key icon), set `DEEPSEEK_API_KEY` and grant notebook access. Never write the key directly into notebook source. DeepSeek calls may incur charges. Any failure stays visible; no synthetic fallback is used.

**Confirm you are authorized** to transmit excerpts of the selected SAP to this model provider.

In [ ]:
from google.colab import userdata
secret = userdata.get("DEEPSEEK_API_KEY")
if not secret:
    raise RuntimeError("Set DEEPSEEK_API_KEY in Colab Secrets and grant access to the notebook.")
llm = DeepSeekJSONClient(model="deepseek-flash", api_key=secret)
del secret
print("Live DeepSeek client initialized (key hidden).")

## 5 · Run selected **Table** targets with real DeepSeek + official Library

For every selected Table: Step 1 parses the real SAP → Step 2 uses live DeepSeek and evidence validation → Step 3 searches the complete **official, approved** Library → Step 4 attempts safe adaptation → Step 5 exports a report. The printed source study, score, review gates, and preview plans help compare easier and harder targets.

**The code does not feed Listing targets into Step 2.** Those are handled by a distinct, retrieval-only cell below.

The previous 0.8108 Demographics score was measured over a smaller source pool; rankings and scores from the consolidated Library are not directly comparable. Scores are heuristic—not statistical probabilities.

In [ ]:
import traceback

sessions = {}
rows = []
for target in TARGET_TABLES:
    tid = target["tfl_id"]
    safe_id = "".join(c if c.isalnum() or c in "._-" else "_" for c in tid)
    print("\n" + "=" * 80)
    print("TARGET:", tid, target["title"])
    session = TFLSession(
        sap_path=SAP_PATH,
        library_path=LIBRARY_PATH,
        schema_path=SCHEMA_PATH,
        prompts_dir=PROMPTS_DIR,
        output_dir=OUTPUT_DIR / safe_id,
        llm=llm,
        tfl_id=tid,
        title=target["title"],
        sap_body_page_range=SAP_BODY_PAGE_RANGE,
        allow_review=False,
    )
    sessions[tid] = session
    try:
        parsed = session.step1_parse()
        # Do NOT use a per-study or synthetic Library: this verified reference
        # contains all approved official templates and their per-study locators.
        session.library = official_library
        assert len(session.library["templates"]) == len(approved_ids)
        print("STEP 1 | SAP blocks:", parsed["block_count"])
        print("STEP 1 | Official approved Library count:", len(session.library["templates"]))

        spec = session.step2_build_specification()
        print("STEP 2 | Specification readiness:", spec["readiness"])
        print("STEP 2 | SAP evidence blocks:", len(spec.get("provenance", {}).get("evidence", [])))
        print("STEP 2 | Schema valid:", session.spec_validation["schema_valid"])
        print("STEP 2 | Completion audit:", (session.spec_debug or {}).get("completion", {}).get("attempted", False))
        display(pd.DataFrame([spec["statistical_requirements"]]).T.rename(columns={0:"Requirement"}))
        if spec.get("issues"):
            display(pd.DataFrame(spec["issues"]).head(12))

        result = session.step3_retrieve_templates(top_k=TOP_K, min_score=MIN_SCORE)
        print("STEP 3 | Retrieval status:", result["status"])
        print("STEP 3 | Best candidate:", result.get("best_candidate_template_id"))
        print("STEP 3 | Automatically eligible:", result.get("selected_template_id"))
        if result.get("reasons"):
            print("STEP 3 | Reasons:", "; ".join(result["reasons"]))
        candidates = result.get("candidates", [])
        if candidates:
            table = pd.DataFrame(candidates)
            selected_columns = [
                col for col in ("template_id", "score", "source_study_id",
                                "table_family", "title_similarity", "statistics_match",
                                "method_similarity", "population_change_required",
                                "requires_endpoint_review", "explanation")
                if col in table.columns
            ]
            display(table[selected_columns])
            best = candidates[0]
            source = index_records[best["template_id"]]
            print("Best-source study:", source["study_id"])
            print("Best-source canonical DOCX:", source["docx_path"])
            print("Best-source locator:", source["locator_map_path"])
            display(FileLink(str((PROJECT_ROOT / source["docx_path"]).resolve()),
                             result_html_prefix="View SOURCE template (not generated): "))
        else:
            print("No eligible output-family candidate found.")

        plan = session.preview_modification_plan()
        print("STEP 4 | Auto-adaptation safe:", plan.get("safe_to_generate", False))
        print("STEP 4 | Preview reason:", plan.get("reason", "No reason reported"))
        generation = session.step4_generate_shell()
        print("STEP 4 | Status:", generation["status"])
        if generation.get("issues"):
            print("STEP 4 | First issues:", generation["issues"][:5])

        report = session.step5_validate_and_export()
        print("STEP 5 | Status:", report["status"])
        if session.shell_path:
            display(FileLink(session.shell_path, result_html_prefix="Generated DRAFT DOCX: "))
        rows.append({
            "Target": tid, "Spec readiness": spec["readiness"],
            "Best candidate": result.get("best_candidate_template_id"),
            "Score": candidates[0]["score"] if candidates else None,
            "Source study": candidates[0].get("source_study_id") if candidates else None,
            "Retrieval status": result["status"], "Generation": report["status"],
            "DOCX generated": bool(session.shell_path),
        })
    except Exception as exc:
        # Never mask API failures, mistaken paths, or schema problems as successes.
        print("FAILED:", type(exc).__name__, str(exc))
        traceback.print_exc()
        rows.append({"Target": tid, "Spec readiness": None, "Best candidate": None,
                     "Score": None, "Source study": None, "Retrieval status": "failed",
                     "Generation": "failed", "DOCX generated": False})

print("\nFINAL RESULTS — OFFICIAL LIBRARY ONLY")
display(pd.DataFrame(rows))
assert all(r["Generation"] != "failed" for r in rows), (
    "At least one target failed. See traceback above before treating this as a completed test."
)

## 6 · Optional two-Listing **retrieval-only** experiment

The target PDF's TFL Content List names **16.2.4.1 Demography Data – Safety Analysis Set** and **16.2.7.1 Adverse Events – Safety Analysis Set** (physical page 25).

Since the current `TFLSession.step2_build_specification()` rejects non-Table outputs, this cell does **not** try to invent a Listing specification or use Step 4. It constructs a minimal title-only search context, then calls the Agent's existing `retrieve_templates` function over approved **Listing** source templates and reports the top matches and source studies. This is exploratory retrieval only. Listing scores are not directly comparable with Table scores that include extracted statistical evidence.

In [ ]:
from tfl_agent.agent.step3_retriever import retrieve_templates

listing_retrieval_report = []
if RUN_LISTING_RETRIEVAL_ONLY:
    approved_listing_count = sum(
        t["identity"]["output_type"] == "listing"
        for t in official_library["templates"]
    )
    print("APPROVED LISTING SOURCE TEMPLATES:", approved_listing_count)
    for target in LISTING_TARGETS:
        # Intentionally sparse title-only retrieval context; NOT an extracted
        # or schema-validated Listing specification.
        sparse_listing_context = {
            "identity": {"tfl_id": target["tfl_id"],
                         "title": target["title"],
                         "output_type": "listing"},
            "statistical_requirements": {
                "analysis_topic": target["title"],
                "endpoint_name": None,
                "analysis_population": [],
                "analysis_methods": [],
                "statistics_to_display": [],
            },
        }
        hits = retrieve_templates(
            sparse_listing_context, official_library, top_k=LISTING_TOP_K
        )
        records = [
            {
                "rank": i,
                "template_id": hit["template_id"],
                "source_study_id": hit["source_study_id"],
                "source_title": hit["template"]["identity"].get(
                    "original_title", ""),
                "heuristic_score": hit["score"],
                "title_similarity": hit["title_similarity"],
                "source_docx": index_records[hit["template_id"]]["docx_path"],
                "locator_map": index_records[hit["template_id"]]["locator_map_path"],
            }
            for i, hit in enumerate(hits, start=1)
        ]
        listing_retrieval_report.append({
            "target_tfl_id": target["tfl_id"],
            "target_title": target["title"],
            "mode": "title_only_listing_retrieval_NO_SPEC_NO_DOCX",
            "approved_listing_source_count": approved_listing_count,
            "candidates": records,
        })
        print("\nLISTING TARGET:", target["tfl_id"], target["title"])
        if records:
            display(pd.DataFrame(records))
        else:
            print("No Listing candidates found.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write_json(
        OUTPUT_DIR / "listing_title_only_retrieval.json",
        {"study_sap_filename": SAP_PATH.name,
         "status": "retrieval_only_not_generation",
         "targets": listing_retrieval_report},
    )
    print("\nSaved Listing retrieval audit to output/official_library_deepseek.")
else:
    print("Listing retrieval experiment disabled.")

## 7 · Export the private audit ZIP

The ZIP includes reports for the selected Table targets and, when enabled, the Listing **retrieval-only** report. It may contain confidential target-SAP excerpts; **do not commit it to public GitHub**. Source templates in `library/` are never edited.

In [ ]:
import shutil
from google.colab import files

assert OUTPUT_DIR.is_dir(), "Run Step 5 first."
archive = Path(shutil.make_archive(
    "/content/tfl_official_library_deepseek_audit", "zip", root_dir=OUTPUT_DIR
))
print("Audit ZIP:", archive)
print("Size (MB):", round(archive.stat().st_size / (1024 * 1024), 2))
files.download(str(archive))

## Interpretation and limitations

- **Source fidelity:** 8 Table targets and 2 Listing targets were verified against the actual `SAP_001_00.pdf` TFL Content List. The SAP body evidence restriction remains **physical pages 4–21** for this file only.
- **Real Table Step 1–3:** DeepSeek extracts SAP evidence, the Agent builds a Table specification, and retrieves source templates from the approved **431-template cross-study Library**. Heuristic scores are not probabilities.
- **Listing retrieval-only:** the two Listing titles are passed to the Agent's existing Step 3 retriever using a **minimal title-only retrieval context**, not a DeepSeek-generated or schema-validated Listing specification. Do not treat this as Listing generation or compare Listing-only scores directly with fully specified Table scores.
- **Step 4:** the bookmark adapter from the separate draft PR is **not yet merged** into `main`. Authentic bookmarked source DOCX files may correctly return `needs_review` with **no new DOCX**.
- **Clinical review:** even source-approved templates require review of target population, denominators, endpoint, column/row structure, footnotes, and clinical meanings before reuse.
- **Other SAPs:** this curated target catalog belongs to `SAP_001_00.pdf` only. For a new PDF, replace the target IDs/titles and verify them against its own Content List.